# Iterative MF + metadata-guided wide-angle destriping

この notebook は、HISUI L1G のメタデータ txt から観測フットプリントの向きを読み取り、
その角度を初期値として `y=x` 系 / `y=-x` 系の stripe 除去を行います。

以前の `row-col=const` 固定では、広い画像で角度が少し違うだけで stripe が残るため、
ここでは `row - slope * col = const`、または角度 `angle_deg` で line grouping を行います。

主なポイント:

- メタデータの `ObservationUpperLeft/Right/LowerLeft/LowerRight` と `MapUpperLeft/Right/LowerLeft` から、観測 line / sample 方向を画像座標へ変換
- `pyproj` がある場合は UTM 座標で計算、ない場合は lon/lat affine の簡易計算へ fallback
- メタデータ由来の角度を探索中心にして、±数度だけ自動探索
- `median`, `sigma_clipped_mean`, `trimmed_mean`, `mean`, `mode` の比較を維持
- `final_only` と `each_iter` の比較を維持


In [ ]:
# ============================================
# 0. Imports and user settings
# ============================================
from __future__ import annotations

import re
from pathlib import Path
from typing import Optional, Sequence, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Input files
# ----------------------------
# ROI CSV: columns = y, x, wave_XXXXnm, wave_XXXXnm, ...
ROI_CSV = Path(r"D:/research/code/all_roi_spectra200x200.csv")

# MODTRAN CSV: columns = wavelength, 0.0, 0.5, 1.0, ...
MODTRAN_CSV = Path(r"E:/refit/CH4c.csv")

OUTPUT_DIR = Path("outputs_metadata_guided_wide_angle_destripe")

# ----------------------------
# Wavelength / UAS settings
# ----------------------------
WL_MIN = 2100.0
WL_MAX = 2450.0
FWHM_NM = 12.5

# Alpha range used to build the UAS template.
UAS_ALPHA_MIN = 0.0
UAS_ALPHA_MAX = 0.5

# ----------------------------
# Iterative MF settings
# ----------------------------
N_ITER = 5
NSIGMA = 3.0
REG = 1e-6
RCOND = 1e-8

# ----------------------------
# Valid pixel settings
# ----------------------------
NODATA_VALUES = [0, -9999]
REQUIRE_POSITIVE = True
MIN_VALID_FRACTION = 1.0

# ----------------------------
# Wide-angle stripe direction settings
# ----------------------------
# Image coordinates:
#   x = column, positive rightward
#   y = row,    positive downward
# angle_deg is measured from +x axis in this image coordinate system.
#   y=x  direction: angle = +45 deg
#   y=-x direction: angle = -45 deg
#
# If the stripe angle is much different from y=x, increase ANGLE_SEARCH_HALF_RANGE_DEG.
# Example: center=45, half_range=30 searches 15 to 75 deg.
INITIAL_LINE_ANGLES_DEG = {
    "y_minus_x": 45.0,
    "y_plus_x": -45.0,
}

# Keep a slope version internally because line grouping is done by
#     row - slope * col = const
# where slope = tan(angle_deg).
def angle_to_slope(angle_deg: float) -> float:
    return float(np.tan(np.deg2rad(angle_deg)))

def slope_to_angle(slope: float) -> float:
    return float(np.rad2deg(np.arctan(float(slope))))

INITIAL_LINE_SLOPES = {
    key: angle_to_slope(angle_deg)
    for key, angle_deg in INITIAL_LINE_ANGLES_DEG.items()
}

# Automatically estimate the best angle from the baseline Iterative MF alpha map.
AUTO_ESTIMATE_LINE_ANGLES = True
AUTO_ESTIMATE_LINE_SLOPES = AUTO_ESTIMATE_LINE_ANGLES  # backwards-compatible alias

# Coarse angle search range.
# If you are unsure, start with 25 or 30 deg.
# For y=x-like stripes, 45±30 deg searches from 15 to 75 deg.
# For y=-x-like stripes, -45±30 deg searches from -75 to -15 deg.
ANGLE_SEARCH_HALF_RANGE_DEG = {
    "y_minus_x": 30.0,
    "y_plus_x": 30.0,
}
ANGLE_SEARCH_STEPS = 121

# Optional second fine search around the coarse best angle.
DO_FINE_ANGLE_SEARCH = True
FINE_ANGLE_SEARCH_HALF_RANGE_DEG = 3.0
FINE_ANGLE_SEARCH_STEPS = 61

# Legacy slope-search parameters are kept but not used when angle search is enabled.
SLOPE_SEARCH_HALF_RANGE = {
    "y_minus_x": 1.2,
    "y_plus_x": 1.2,
}
SLOPE_SEARCH_STEPS = 121

# Width of the line-coordinate bin in row-intercept units.
# 1.0 is usually the first choice. If the residual stripe is slightly split,
# try 1.5 or 2.0. Larger values make the correction stronger and less local.
LINE_BIN_WIDTH = 1.0

# ----------------------------
# Recommended destriping parameters
# ----------------------------
DEFAULT_DESTRIPE_PARAMS = {
    # Correction directions.
    # y=x stripes:  "y_minus_x" -> row - col = const
    # y=-x stripes: "y_plus_x"  -> row + col = const
    # This default corrects y=x first, then y=-x.
    "directions": ["y_minus_x", "y_plus_x"],

    # Slopes used for each direction. These may be overwritten by automatic slope estimation.
    "line_angles_deg": INITIAL_LINE_ANGLES_DEG.copy(),
    "line_slopes": INITIAL_LINE_SLOPES.copy(),
    "line_bin_width": LINE_BIN_WIDTH,

    # Parameters for automatic slope estimation.
    "auto_estimate_line_angles": AUTO_ESTIMATE_LINE_ANGLES,
    "auto_estimate_line_slopes": AUTO_ESTIMATE_LINE_SLOPES,
    "angle_search_half_range_deg": ANGLE_SEARCH_HALF_RANGE_DEG,
    "angle_search_steps": ANGLE_SEARCH_STEPS,
    "do_fine_angle_search": DO_FINE_ANGLE_SEARCH,
    "fine_angle_search_half_range_deg": FINE_ANGLE_SEARCH_HALF_RANGE_DEG,
    "fine_angle_search_steps": FINE_ANGLE_SEARCH_STEPS,
    "slope_search_half_range": SLOPE_SEARCH_HALF_RANGE,
    "slope_search_steps": SLOPE_SEARCH_STEPS,
    # For slope estimation, usually do not exclude high-alpha pixels because the stripe itself
    # may be high alpha. Use "robust_high" only if real plume dominates the slope estimate.
    "slope_estimate_exclude_mode": "none",

    # "median", "mean", "trimmed_mean", "mode", "sigma_clipped_mean"
    "method": "median",

    # Minimum pixels per diagonal line for estimating an offset.
    "min_pixels_per_line": 5,

    # True: subtract line_stat - global_stat to preserve the global alpha baseline.
    # False: subtract line_stat directly.
    "preserve_global_stat": True,

    # Half-window for median smoothing of neighboring diagonal offsets. Try 0, 1, or 2.
    "smooth_half_window": 0,

    # Exclusion mode: "none", "robust_high", "previous_plume", "previous_plume_or_high".
    # If the stripe artifact itself enters the plume candidates, "robust_high" or "none"
    # can work better than "previous_plume".
    "exclude_mode": "robust_high",
    "exclude_nsigma": NSIGMA,

    # Recompute the exclude mask before estimating the second direction.
    "recompute_exclude_each_direction": True,

    # If too few pixels remain after exclusion, re-estimate from all valid pixels.
    "fallback_to_valid": True,

    # For trimmed_mean.
    "trim_fraction": 0.1,

    # For mode. Continuous alpha values use a histogram approximation.
    "mode_bins": 64,

    # For sigma_clipped_mean. Clip with median + MAD, then average.
    "sigma_clip_nsigma": 3.0,
    "sigma_clip_max_iter": 3,

    # For each_iter, choose whether plume thresholding uses corrected or raw alpha.
    "threshold_source": "corrected",
}

# ----------------------------
# Experiment settings
# ----------------------------
# destripe_when:
#   "none"       : no destriping
#   "final_only" : destripe only the final alpha map after N_ITER
#   "each_iter"  : destripe before plume thresholding at every iteration
#   [3,4,5]      : destripe only at selected iteration numbers

# Every statistic listed here is run as both final_only and each_iter.
STRIPE_STAT_METHODS = [
    ("median", {}),
    ("mean", {}),
    ("trimmed_mean", {"trim_fraction": 0.1}),
    ("mode", {"mode_bins": 64}),
    ("sigma_clipped_mean", {"sigma_clip_nsigma": 3.0, "sigma_clip_max_iter": 3}),
]

EXPERIMENTS = {
    "baseline_no_destripe": {
        "destripe_when": "none",
        "destripe_params": None,
    },

    # y=x only: useful to compare against the two-direction correction.
    "median_yx_final_only": {
        "destripe_when": "final_only",
        "destripe_params": {**DEFAULT_DESTRIPE_PARAMS, "directions": ["y_minus_x"], "method": "median"},
    },
    "median_yx_each_iter": {
        "destripe_when": "each_iter",
        "destripe_params": {**DEFAULT_DESTRIPE_PARAMS, "directions": ["y_minus_x"], "method": "median"},
    },
}

# Two-direction correction, y=x -> y=-x, for every statistic and timing mode.
for method, overrides in STRIPE_STAT_METHODS:
    for when in ("final_only", "each_iter"):
        EXPERIMENTS[f"{method}_yx_then_ynegx_{when}"] = {
            "destripe_when": when,
            "destripe_params": {**DEFAULT_DESTRIPE_PARAMS, "method": method, **overrides},
        }

# Optional comparison: destripe only after iteration 3.
EXPERIMENTS["median_yx_then_ynegx_iter_3_to_5"] = {
    "destripe_when": [3, 4, 5],
    "destripe_params": {**DEFAULT_DESTRIPE_PARAMS, "method": "median"},
}




In [ ]:

# ============================================
# 1. Data loading helpers
# ============================================

def get_wave_columns(df: pd.DataFrame) -> tuple[list[str], np.ndarray]:
    """Return wavelength columns sorted by wavelength."""
    wave_cols: list[str] = []
    wavelengths: list[float] = []

    pattern = re.compile(r"^wave_([0-9]+(?:\.[0-9]+)?)nm$")
    for col in df.columns:
        match = pattern.match(col)
        if match is not None:
            wave_cols.append(col)
            wavelengths.append(float(match.group(1)))

    if len(wave_cols) == 0:
        raise ValueError("No wavelength columns like wave_2300nm were found.")

    wavelengths_arr = np.asarray(wavelengths, dtype=float)
    order = np.argsort(wavelengths_arr)
    wavelengths_arr = wavelengths_arr[order]
    wave_cols_sorted = [wave_cols[i] for i in order]
    return wave_cols_sorted, wavelengths_arr


def load_roi_spectra_csv(path: str | Path) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """Load ROI spectra CSV with y, x, and wave_XXXXnm columns."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"ROI CSV not found: {path}")

    df = pd.read_csv(path)
    if "y" not in df.columns or "x" not in df.columns:
        raise ValueError("ROI CSV must contain columns 'y' and 'x'.")

    wave_cols, wavelengths = get_wave_columns(df)
    spectra = df[wave_cols].to_numpy(dtype=float)
    return df, wavelengths, spectra


def spectra_to_cube(
    df: pd.DataFrame,
    spectra: np.ndarray,
    fill_value: float = np.nan,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Convert table spectra to image cube [H, W, B]."""
    ys = np.sort(df["y"].unique())
    xs = np.sort(df["x"].unique())

    y_to_row = {y: i for i, y in enumerate(ys)}
    x_to_col = {x: j for j, x in enumerate(xs)}

    H, W = len(ys), len(xs)
    B = spectra.shape[1]
    cube = np.full((H, W, B), fill_value, dtype=float)

    for row_idx, row in df.iterrows():
        r = y_to_row[row["y"]]
        c = x_to_col[row["x"]]
        cube[r, c, :] = spectra[row_idx, :]

    return cube, ys, xs


def band_mask(
    wavelengths: np.ndarray,
    wl_min: Optional[float] = None,
    wl_max: Optional[float] = None,
    exclude_ranges: Optional[Sequence[tuple[float, float]]] = None,
) -> np.ndarray:
    mask = np.ones_like(wavelengths, dtype=bool)

    if wl_min is not None:
        mask &= wavelengths >= wl_min
    if wl_max is not None:
        mask &= wavelengths <= wl_max

    if exclude_ranges is not None:
        for a, b in exclude_ranges:
            mask &= ~((wavelengths >= a) & (wavelengths <= b))

    return mask


def select_bands(
    cube: np.ndarray,
    wavelengths: np.ndarray,
    wl_min: float,
    wl_max: float,
    exclude_ranges: Optional[Sequence[tuple[float, float]]] = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    mask = band_mask(wavelengths, wl_min=wl_min, wl_max=wl_max, exclude_ranges=exclude_ranges)
    return cube[:, :, mask], wavelengths[mask], mask


In [ ]:

# ============================================
# 2. Plotting helpers
# ============================================

def finite_values(img: np.ndarray, mask: Optional[np.ndarray] = None) -> np.ndarray:
    arr = np.asarray(img, dtype=float)
    if mask is None:
        vals = arr[np.isfinite(arr)]
    else:
        vals = arr[np.isfinite(arr) & np.asarray(mask, dtype=bool)]
    return vals


def robust_limits(
    images: Union[np.ndarray, Sequence[np.ndarray]],
    mask: Optional[np.ndarray] = None,
    q_low: float = 2,
    q_high: float = 98,
) -> tuple[float, float]:
    if isinstance(images, np.ndarray):
        images = [images]

    vals_list = []
    for img in images:
        vals = finite_values(img, mask=mask)
        if vals.size > 0:
            vals_list.append(vals)

    if len(vals_list) == 0:
        return -1.0, 1.0

    vals_all = np.concatenate(vals_list)
    lo, hi = np.nanpercentile(vals_all, [q_low, q_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.nanmin(vals_all)), float(np.nanmax(vals_all))
        if hi <= lo:
            hi = lo + 1.0
    return float(lo), float(hi)


def robust_scale_image(img: np.ndarray, pmin: float = 2, pmax: float = 98) -> np.ndarray:
    vals = finite_values(img)
    if vals.size == 0:
        return np.zeros_like(img, dtype=float)
    lo, hi = np.nanpercentile(vals, [pmin, pmax])
    if hi <= lo:
        return np.zeros_like(img, dtype=float)
    return np.clip((img - lo) / (hi - lo), 0, 1)


def make_rgb_from_cube(
    cube: np.ndarray,
    wavelengths: np.ndarray,
    r_wl: float = 650,
    g_wl: float = 550,
    b_wl: float = 460,
) -> np.ndarray:
    idx_r = int(np.argmin(np.abs(wavelengths - r_wl)))
    idx_g = int(np.argmin(np.abs(wavelengths - g_wl)))
    idx_b = int(np.argmin(np.abs(wavelengths - b_wl)))

    r = robust_scale_image(cube[:, :, idx_r])
    g = robust_scale_image(cube[:, :, idx_g])
    b = robust_scale_image(cube[:, :, idx_b])
    return np.dstack([r, g, b])


def plot_map(
    img: np.ndarray,
    title: str = "Map",
    mask: Optional[np.ndarray] = None,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    cmap: str = "viridis",
    colorbar_label: Optional[str] = None,
    figsize: tuple[float, float] = (5, 5),
):
    if vmin is None or vmax is None:
        auto_vmin, auto_vmax = robust_limits(img, mask=mask)
        if vmin is None:
            vmin = auto_vmin
        if vmax is None:
            vmax = auto_vmax

    plt.figure(figsize=figsize)
    im = plt.imshow(img, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.colorbar(im, label=colorbar_label)
    plt.show()


def plot_mean_spectrum(cube: np.ndarray, wavelengths: np.ndarray, mask: Optional[np.ndarray] = None, xlim=None):
    if mask is None:
        X = cube.reshape(-1, cube.shape[2])
        spec = np.nanmean(X, axis=0)
    else:
        spec = np.nanmean(cube[mask], axis=0)

    plt.figure(figsize=(8, 4))
    plt.plot(wavelengths, spec, marker="o", ms=3)
    plt.xlabel("Wavelength [nm]")
    plt.ylabel("Radiance / Reflectance")
    plt.title("Mean spectrum")
    if xlim is not None:
        plt.xlim(*xlim)
    plt.grid(True)
    plt.show()


def plot_uas(wavelengths: np.ndarray, uas: np.ndarray, title: str = "UAS", xlim=None):
    plt.figure(figsize=(8, 4))
    plt.plot(wavelengths, uas, marker="o", ms=3)
    plt.xlabel("Wavelength [nm]")
    plt.ylabel("UAS")
    plt.title(title)
    if xlim is not None:
        plt.xlim(*xlim)
    plt.grid(True)
    plt.show()


In [ ]:

# ============================================
# 3. MODTRAN / UAS helpers
# ============================================

def load_ch4_modtran_csv(path: str | Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Load MODTRAN spectra CSV.

    Expected format:
        wavelength, 0.0, 0.5, 1.0, ...

    Returns:
        mod_wave: (n_mod_wave,)
        alpha_grid: (n_alpha,)
        spectra_grid: (n_alpha, n_mod_wave)
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"MODTRAN CSV not found: {path}")

    df_mod = pd.read_csv(path)
    if "wavelength" not in df_mod.columns:
        raise ValueError("MODTRAN CSV must contain a 'wavelength' column.")

    mod_wave = df_mod["wavelength"].to_numpy(dtype=float)
    alpha_cols = [c for c in df_mod.columns if c != "wavelength"]

    try:
        alpha_grid = np.array([float(c) for c in alpha_cols], dtype=float)
    except ValueError as exc:
        raise ValueError("MODTRAN columns other than 'wavelength' must be numeric alpha values.") from exc

    order = np.argsort(alpha_grid)
    alpha_grid = alpha_grid[order]
    alpha_cols = [alpha_cols[i] for i in order]

    spectra_grid = df_mod[alpha_cols].to_numpy(dtype=float).T
    return mod_wave, alpha_grid, spectra_grid


def gaussian_srf_resample(
    mod_wave: np.ndarray,
    mod_spectra: np.ndarray,
    sensor_wave: np.ndarray,
    fwhm_nm: Union[float, np.ndarray],
) -> np.ndarray:
    """Resample high-resolution MODTRAN spectra to sensor wavelengths by Gaussian SRF."""
    mod_wave = np.asarray(mod_wave, dtype=float)
    mod_spectra = np.asarray(mod_spectra, dtype=float)
    sensor_wave = np.asarray(sensor_wave, dtype=float)

    if np.isscalar(fwhm_nm):
        fwhm_arr = np.full(sensor_wave.shape, float(fwhm_nm), dtype=float)
    else:
        fwhm_arr = np.asarray(fwhm_nm, dtype=float)
        if fwhm_arr.shape != sensor_wave.shape:
            raise ValueError("fwhm_nm must be scalar or same shape as sensor_wave.")

    out = np.full((mod_spectra.shape[0], sensor_wave.size), np.nan, dtype=float)

    for j, center in enumerate(sensor_wave):
        sigma = fwhm_arr[j] / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        use = np.abs(mod_wave - center) <= 4.0 * sigma

        if np.sum(use) < 2:
            # fallback: simple interpolation for each alpha spectrum
            for i in range(mod_spectra.shape[0]):
                out[i, j] = np.interp(center, mod_wave, mod_spectra[i])
            continue

        weights = np.exp(-0.5 * ((mod_wave[use] - center) / sigma) ** 2)
        weights = weights / np.sum(weights)
        out[:, j] = mod_spectra[:, use] @ weights

    return out


def compute_uas_log_slope(
    alpha_grid: np.ndarray,
    spectra_grid: np.ndarray,
    alpha_min: Optional[float] = None,
    alpha_max: Optional[float] = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute UAS by fitting log(spectrum) = intercept - UAS * alpha."""
    alpha_grid = np.asarray(alpha_grid, dtype=float)
    spectra_grid = np.asarray(spectra_grid, dtype=float)

    use = np.ones_like(alpha_grid, dtype=bool)
    if alpha_min is not None:
        use &= alpha_grid >= alpha_min
    if alpha_max is not None:
        use &= alpha_grid <= alpha_max

    a = alpha_grid[use]
    if a.size < 2:
        raise ValueError("Need at least two alpha values to compute UAS.")

    Y = np.log(np.maximum(spectra_grid[use], 1e-30))
    A = np.vstack([np.ones_like(a), a]).T
    coeff, _, _, _ = np.linalg.lstsq(A, Y, rcond=None)

    intercept = coeff[0]
    slope = coeff[1]
    uas = -slope
    return uas, intercept


In [ ]:

# ============================================
# 4. Matched Filter helpers
# ============================================

def make_valid_pixel_mask(
    cube: np.ndarray,
    nodata_values: Optional[Sequence[float]] = None,
    require_positive: bool = True,
    min_valid_fraction: float = 1.0,
) -> np.ndarray:
    valid_band = np.isfinite(cube)

    if nodata_values is not None:
        for value in nodata_values:
            valid_band &= cube != value

    if require_positive:
        valid_band &= cube > 0

    valid_fraction = np.mean(valid_band, axis=2)
    return valid_fraction >= min_valid_fraction


def estimate_background_mean_cov_from_cube(
    cube: np.ndarray,
    background_mask: np.ndarray,
    reg: float = 1e-6,
    min_pixels: Optional[int] = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Estimate background mean and covariance from selected pixels."""
    H, W, B = cube.shape

    if min_pixels is None:
        min_pixels = max(B + 5, 30)

    X = cube[np.asarray(background_mask, dtype=bool)]
    X = X[np.all(np.isfinite(X), axis=1)]

    if X.shape[0] < min_pixels:
        raise ValueError(f"Too few background pixels: {X.shape[0]} < {min_pixels}")

    mu = np.mean(X, axis=0)
    Xc = X - mu
    cov = (Xc.T @ Xc) / max(X.shape[0] - 1, 1)

    # A small diagonal regularization. Scale-aware enough for most reflectance/radiance cases.
    scale = np.nanmean(np.diag(cov))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    cov = cov + reg * scale * np.eye(B)

    return mu, cov, X


def make_methane_target(mu: np.ndarray, uas: np.ndarray, positive_alpha: bool = True) -> np.ndarray:
    """MF target spectrum. positive_alpha=True follows the convention used in the original notebook."""
    mu = np.asarray(mu, dtype=float).reshape(-1)
    uas = np.asarray(uas, dtype=float).reshape(-1)
    return -mu * uas if positive_alpha else mu * uas


def matched_filter_alpha_map(
    cube: np.ndarray,
    uas: np.ndarray,
    valid_mask: np.ndarray,
    background_mask: np.ndarray,
    reg: float = 1e-6,
    rcond: float = 1e-8,
    min_background_pixels: Optional[int] = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Compute MF alpha map using current background pixels."""
    H, W, B = cube.shape
    uas = np.asarray(uas, dtype=float).reshape(-1)
    if uas.size != B:
        raise ValueError(f"uas length {uas.size} does not match cube bands {B}.")

    valid_mask = np.asarray(valid_mask, dtype=bool)
    background_mask = valid_mask & np.asarray(background_mask, dtype=bool)

    mu, cov, _ = estimate_background_mean_cov_from_cube(
        cube=cube,
        background_mask=background_mask,
        reg=reg,
        min_pixels=min_background_pixels,
    )

    target = make_methane_target(mu, uas, positive_alpha=True)
    inv_cov = np.linalg.pinv(cov, rcond=rcond)

    denom = float(target.T @ inv_cov @ target)
    if abs(denom) < 1e-15:
        raise ValueError("MF denominator is too small. Check UAS, covariance, and wavelength selection.")

    alpha_map = np.full((H, W), np.nan, dtype=float)
    X = cube[valid_mask]
    diff = X - mu
    alpha_values = (diff @ inv_cov @ target) / denom
    alpha_map[valid_mask] = alpha_values

    return alpha_map, mu, cov, target


def robust_threshold_from_alpha(alpha_values: np.ndarray, nsigma: float = 4.0) -> tuple[float, float, float]:
    """Median + nsigma * 1.4826*MAD threshold."""
    values = np.asarray(alpha_values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError("No finite alpha values for thresholding.")

    med = float(np.nanmedian(values))
    mad = float(np.nanmedian(np.abs(values - med)))
    robust_std = 1.4826 * mad + 1e-12
    threshold = med + nsigma * robust_std
    return float(threshold), med, float(robust_std)


def plume_mask_from_alpha(
    alpha_map: np.ndarray,
    valid_mask: np.ndarray,
    nsigma: float = 4.0,
) -> tuple[np.ndarray, dict]:
    threshold, med, robust_std = robust_threshold_from_alpha(alpha_map[valid_mask], nsigma=nsigma)
    plume_mask = np.zeros_like(valid_mask, dtype=bool)
    plume_mask[valid_mask] = alpha_map[valid_mask] > threshold

    meta = {
        "threshold": threshold,
        "median": med,
        "robust_std": robust_std,
        "n_plume": int(np.sum(plume_mask)),
    }
    return plume_mask, meta


In [ ]:
# ============================================
# 5. Improved near-diagonal / oriented destriping helpers
# ============================================

def default_slope_for_direction(direction: str) -> float:
    """Default row-vs-col slope for the named stripe direction."""
    if direction == "y_minus_x":
        return 1.0
    if direction == "y_plus_x":
        return -1.0
    raise ValueError("direction must be 'y_minus_x' or 'y_plus_x'.")


def get_direction_param(params: Optional[dict], key: str, direction: str, default=None):
    """Read a parameter that may be scalar or a dict keyed by direction."""
    if params is None or key not in params:
        return default
    value = params[key]
    if isinstance(value, dict):
        return value.get(direction, default)
    return value


def angle_to_slope_runtime(angle_deg: float) -> float:
    """Convert image-coordinate angle in degrees to row-vs-col slope."""
    return float(np.tan(np.deg2rad(angle_deg)))


def slope_to_angle_runtime(slope: float) -> float:
    """Convert row-vs-col slope to image-coordinate angle in degrees."""
    return float(np.rad2deg(np.arctan(float(slope))))


def get_line_angle(params: Optional[dict], direction: str) -> Optional[float]:
    angles = None if params is None else params.get("line_angles_deg", None)
    if isinstance(angles, dict):
        if direction in angles:
            return float(angles[direction])
    if angles is not None and np.isscalar(angles):
        return float(angles)
    return None


def get_line_slope(params: Optional[dict], direction: str) -> float:
    # Prefer angle if supplied, because wide-angle search is easier to interpret in degrees.
    angle = get_line_angle(params, direction)
    if angle is not None:
        return angle_to_slope_runtime(angle)

    slopes = None if params is None else params.get("line_slopes", None)
    if isinstance(slopes, dict):
        return float(slopes.get(direction, default_slope_for_direction(direction)))
    if slopes is not None and np.isscalar(slopes):
        return float(slopes)
    return default_slope_for_direction(direction)


def normalize_directions(destripe_params: Optional[dict]) -> list[str]:
    """Accept either 'directions' or old-style 'direction'."""
    if destripe_params is None:
        return []

    if "directions" in destripe_params:
        dirs = destripe_params["directions"]
    else:
        dirs = destripe_params.get("direction", "y_minus_x")

    if isinstance(dirs, str):
        dirs = [dirs]

    dirs = list(dirs)
    for d in dirs:
        if d not in {"y_minus_x", "y_plus_x"}:
            raise ValueError("directions must contain only 'y_minus_x' and/or 'y_plus_x'.")
    return dirs


def line_coordinate_map(
    shape: tuple[int, int],
    direction: str = "y_minus_x",
    slope: Optional[float] = None,
) -> np.ndarray:
    """Return continuous line coordinate.

    We model stripe lines as
        row - slope * col = const

    For direction='y_minus_x': default slope = +1  -> row - col = const.
    For direction='y_plus_x' : default slope = -1  -> row + col = const.

    If the wide image shows a 1 px drift over 1500 px, use slope=1±1/1500
    for y=x-like stripes.
    """
    rows, cols = np.indices(shape, dtype=float)
    if slope is None:
        slope = default_slope_for_direction(direction)
    return rows - float(slope) * cols


def line_id_map(
    shape: tuple[int, int],
    direction: str = "y_minus_x",
    slope: Optional[float] = None,
    line_bin_width: float = 1.0,
) -> np.ndarray:
    """Return integer line IDs for near-diagonal line grouping.

    line_bin_width controls how continuous line coordinates are quantized.
    1.0 is the one-pixel default. Try 1.5 or 2.0 if the stripe is slightly thick.
    """
    if line_bin_width <= 0:
        raise ValueError("line_bin_width must be positive.")
    coord = line_coordinate_map(shape, direction=direction, slope=slope)
    return np.rint(coord / float(line_bin_width)).astype(np.int64)


def moving_nanmedian_1d(values: np.ndarray, half_window: int = 0) -> np.ndarray:
    """Median-smooth a 1D array while ignoring NaN."""
    values = np.asarray(values, dtype=float)
    if half_window <= 0:
        return values.copy()

    out = np.full_like(values, np.nan, dtype=float)
    for i in range(values.size):
        lo = max(0, i - half_window)
        hi = min(values.size, i + half_window + 1)
        win = values[lo:hi]
        finite = np.isfinite(win)
        if np.any(finite):
            out[i] = np.nanmedian(win[finite])
    return out


def statistic_1d(
    values: np.ndarray,
    method: str = "median",
    trim_fraction: float = 0.1,
    mode_bins: int = 64,
    sigma_clip_nsigma: float = 3.0,
    sigma_clip_max_iter: int = 3,
    quantile_q: float = 0.5,
) -> float:
    """Compute 1D statistic for stripe offset estimation."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    if method == "median":
        return float(np.nanmedian(values))

    if method == "mean":
        return float(np.nanmean(values))

    if method == "trimmed_mean":
        v = np.sort(values)
        k = int(np.floor(trim_fraction * v.size))
        if 2 * k >= v.size:
            return float(np.nanmean(v))
        return float(np.nanmean(v[k:v.size-k]))

    if method == "winsorized_mean":
        lo, hi = np.nanpercentile(values, [100 * trim_fraction, 100 * (1 - trim_fraction)])
        v = np.clip(values, lo, hi)
        return float(np.nanmean(v))

    if method == "quantile":
        return float(np.nanquantile(values, quantile_q))

    if method == "mode":
        # Continuous values do not have an exact mode, so use a robust histogram approximation.
        if values.size == 1 or np.allclose(values, values[0]):
            return float(values[0])
        # Avoid extreme outliers widening the histogram range.
        lo, hi = np.nanpercentile(values, [1, 99])
        v = values[(values >= lo) & (values <= hi)]
        if v.size < 3 or not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            v = values
        counts, edges = np.histogram(v, bins=mode_bins)
        idx = int(np.argmax(counts))
        return float(0.5 * (edges[idx] + edges[idx + 1]))

    if method == "sigma_clipped_mean":
        clipped = values.copy()
        for _ in range(int(sigma_clip_max_iter)):
            if clipped.size < 3:
                break
            med = float(np.nanmedian(clipped))
            mad = float(np.nanmedian(np.abs(clipped - med)))
            robust_std = 1.4826 * mad

            if not np.isfinite(robust_std) or robust_std <= 0:
                # MAD can be zero for nearly constant lines. Fall back to ordinary std.
                std = float(np.nanstd(clipped))
                if not np.isfinite(std) or std <= 0:
                    break
                keep = np.abs(clipped - med) <= sigma_clip_nsigma * std
            else:
                keep = np.abs(clipped - med) <= sigma_clip_nsigma * robust_std

            if np.all(keep) or not np.any(keep):
                break
            clipped = clipped[keep]
        return float(np.nanmean(clipped))

    raise ValueError(
        "method must be 'median', 'mean', 'trimmed_mean', 'winsorized_mean', "
        "'quantile', 'mode', or 'sigma_clipped_mean'."
    )


def grouped_line_stats(
    alpha: np.ndarray,
    ids: np.ndarray,
    mask: np.ndarray,
    method: str = "median",
    trim_fraction: float = 0.1,
    mode_bins: int = 64,
    sigma_clip_nsigma: float = 3.0,
    sigma_clip_max_iter: int = 3,
    quantile_q: float = 0.5,
) -> pd.DataFrame:
    """Efficiently compute a statistic for each integer line ID.

    This avoids looping over all lines with a full image-size boolean mask, which is slow
    for 1500x1500 images.
    """
    alpha = np.asarray(alpha, dtype=float)
    ids = np.asarray(ids)
    mask = np.asarray(mask, dtype=bool) & np.isfinite(alpha)

    flat_ids = ids.ravel()[mask.ravel()]
    flat_values = alpha.ravel()[mask.ravel()]

    if flat_values.size == 0:
        return pd.DataFrame(columns=["line_id", "n_pixels", "line_stat"])

    order = np.argsort(flat_ids, kind="mergesort")
    ids_sorted = flat_ids[order]
    values_sorted = flat_values[order]

    unique_ids, starts, counts = np.unique(ids_sorted, return_index=True, return_counts=True)
    stats = np.full(unique_ids.shape, np.nan, dtype=float)

    for i, (s, c) in enumerate(zip(starts, counts)):
        stats[i] = statistic_1d(
            values_sorted[s:s+c],
            method=method,
            trim_fraction=trim_fraction,
            mode_bins=mode_bins,
            sigma_clip_nsigma=sigma_clip_nsigma,
            sigma_clip_max_iter=sigma_clip_max_iter,
            quantile_q=quantile_q,
        )

    return pd.DataFrame({
        "line_id": unique_ids.astype(np.int64),
        "n_pixels": counts.astype(int),
        "line_stat": stats,
    })


def make_exclude_mask_for_destriping(
    alpha_map: np.ndarray,
    valid_mask: np.ndarray,
    plume_mask: Optional[np.ndarray] = None,
    exclude_mode: str = "robust_high",
    exclude_nsigma: float = 4.0,
) -> tuple[np.ndarray, dict]:
    """Build mask of pixels excluded from stripe estimation."""
    valid_mask = np.asarray(valid_mask, dtype=bool) & np.isfinite(alpha_map)
    exclude = np.zeros_like(valid_mask, dtype=bool)
    meta: dict = {"exclude_mode": exclude_mode}

    if exclude_mode == "none" or exclude_mode is None:
        meta["n_excluded"] = 0
        return exclude, meta

    if "high" in exclude_mode:
        threshold, med, robust_std = robust_threshold_from_alpha(alpha_map[valid_mask], nsigma=exclude_nsigma)
        high_mask = np.zeros_like(valid_mask, dtype=bool)
        high_mask[valid_mask] = alpha_map[valid_mask] > threshold
        exclude |= high_mask
        meta.update({
            "high_threshold": float(threshold),
            "high_median": float(med),
            "high_robust_std": float(robust_std),
            "n_high_excluded": int(np.sum(high_mask)),
        })

    if "plume" in exclude_mode:
        if plume_mask is not None:
            plume_mask = np.asarray(plume_mask, dtype=bool)
            exclude |= plume_mask
            meta["n_plume_excluded"] = int(np.sum(plume_mask))
        else:
            meta["n_plume_excluded"] = 0

    if exclude_mode not in {"robust_high", "previous_plume", "previous_plume_or_high", "none", None}:
        raise ValueError(
            "exclude_mode must be 'none', 'robust_high', 'previous_plume', or 'previous_plume_or_high'."
        )

    exclude &= valid_mask
    meta["n_excluded"] = int(np.sum(exclude))
    return exclude, meta


def weighted_robust_std(values: np.ndarray, weights: Optional[np.ndarray] = None) -> float:
    """Simple robust spread used as a slope-alignment score."""
    values = np.asarray(values, dtype=float)
    good = np.isfinite(values)
    values = values[good]
    if weights is not None:
        weights = np.asarray(weights, dtype=float)[good]
    if values.size < 3:
        return np.nan
    med = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values - med))
    return float(1.4826 * mad)


def estimate_best_slope_for_direction(
    alpha_map: np.ndarray,
    valid_mask: np.ndarray,
    direction: str,
    base_slope: float,
    search_half_range: float,
    search_steps: int = 41,
    line_bin_width: float = 1.0,
    method: str = "median",
    min_pixels_per_line: int = 5,
    exclude_mask: Optional[np.ndarray] = None,
    trim_fraction: float = 0.1,
    mode_bins: int = 64,
    sigma_clip_nsigma: float = 3.0,
    sigma_clip_max_iter: int = 3,
    quantile_q: float = 0.5,
) -> tuple[float, pd.DataFrame]:
    """Estimate near-diagonal slope by maximizing line-offset contrast.

    If the slope is correct, one physical stripe stays within the same line group,
    so line-to-line offset contrast becomes large. If the slope is wrong, a stripe is
    split among neighboring line IDs and the contrast is weakened.
    """
    if search_steps <= 1 or search_half_range <= 0:
        slopes = np.array([base_slope], dtype=float)
    else:
        slopes = np.linspace(base_slope - search_half_range, base_slope + search_half_range, int(search_steps))

    alpha = np.asarray(alpha_map, dtype=float)
    valid = np.asarray(valid_mask, dtype=bool) & np.isfinite(alpha)
    if exclude_mask is None:
        estimate_mask = valid
    else:
        estimate_mask = valid & (~np.asarray(exclude_mask, dtype=bool))

    global_stat = statistic_1d(
        alpha[estimate_mask],
        method=method,
        trim_fraction=trim_fraction,
        mode_bins=mode_bins,
        sigma_clip_nsigma=sigma_clip_nsigma,
        sigma_clip_max_iter=sigma_clip_max_iter,
        quantile_q=quantile_q,
    )

    rows = []
    for slope in slopes:
        ids = line_id_map(alpha.shape, direction=direction, slope=slope, line_bin_width=line_bin_width)
        stats_df = grouped_line_stats(
            alpha=alpha,
            ids=ids,
            mask=estimate_mask,
            method=method,
            trim_fraction=trim_fraction,
            mode_bins=mode_bins,
            sigma_clip_nsigma=sigma_clip_nsigma,
            sigma_clip_max_iter=sigma_clip_max_iter,
            quantile_q=quantile_q,
        )
        if len(stats_df) == 0:
            score = np.nan
            n_good = 0
            p95_abs_offset = np.nan
        else:
            good = (stats_df["n_pixels"].to_numpy() >= min_pixels_per_line) & np.isfinite(stats_df["line_stat"].to_numpy())
            offsets = stats_df["line_stat"].to_numpy(dtype=float)[good] - global_stat
            counts = stats_df["n_pixels"].to_numpy(dtype=float)[good]
            n_good = int(np.sum(good))
            score = weighted_robust_std(offsets, weights=counts)
            p95_abs_offset = float(np.nanpercentile(np.abs(offsets), 95)) if offsets.size > 0 else np.nan

        rows.append({
            "direction": direction,
            "slope": float(slope),
            "score_robust_std_of_line_offsets": score,
            "p95_abs_line_offset": p95_abs_offset,
            "n_good_lines": n_good,
            "global_stat": float(global_stat),
        })

    score_df = pd.DataFrame(rows)
    if np.all(~np.isfinite(score_df["score_robust_std_of_line_offsets"].to_numpy())):
        best_slope = float(base_slope)
    else:
        idx = int(np.nanargmax(score_df["score_robust_std_of_line_offsets"].to_numpy()))
        best_slope = float(score_df.iloc[idx]["slope"])
    return best_slope, score_df



def estimate_best_angle_for_direction(
    alpha_map: np.ndarray,
    valid_mask: np.ndarray,
    direction: str,
    base_angle_deg: float,
    search_half_range_deg: float,
    search_steps: int = 121,
    line_bin_width: float = 1.0,
    method: str = "median",
    min_pixels_per_line: int = 5,
    exclude_mask: Optional[np.ndarray] = None,
    trim_fraction: float = 0.1,
    mode_bins: int = 64,
    sigma_clip_nsigma: float = 3.0,
    sigma_clip_max_iter: int = 3,
    quantile_q: float = 0.5,
) -> tuple[float, float, pd.DataFrame]:
    """Estimate stripe angle by maximizing line-offset contrast.

    angle_deg is measured from +x axis in image coordinates, where y=row is positive downward.
    y=x corresponds to +45 deg, and y=-x corresponds to -45 deg.
    """
    if search_steps <= 1 or search_half_range_deg <= 0:
        angles = np.array([base_angle_deg], dtype=float)
    else:
        angles = np.linspace(
            base_angle_deg - search_half_range_deg,
            base_angle_deg + search_half_range_deg,
            int(search_steps),
        )

    # Avoid angles too close to ±90 deg because tan(angle) explodes.
    angles = angles[np.abs(np.cos(np.deg2rad(angles))) > 1e-3]
    if angles.size == 0:
        angles = np.array([base_angle_deg], dtype=float)

    alpha = np.asarray(alpha_map, dtype=float)
    valid = np.asarray(valid_mask, dtype=bool) & np.isfinite(alpha)
    if exclude_mask is None:
        estimate_mask = valid
    else:
        estimate_mask = valid & (~np.asarray(exclude_mask, dtype=bool))

    global_stat = statistic_1d(
        alpha[estimate_mask],
        method=method,
        trim_fraction=trim_fraction,
        mode_bins=mode_bins,
        sigma_clip_nsigma=sigma_clip_nsigma,
        sigma_clip_max_iter=sigma_clip_max_iter,
        quantile_q=quantile_q,
    )

    rows = []
    for angle_deg in angles:
        slope = angle_to_slope_runtime(float(angle_deg))
        ids = line_id_map(alpha.shape, direction=direction, slope=slope, line_bin_width=line_bin_width)
        stats_df = grouped_line_stats(
            alpha=alpha,
            ids=ids,
            mask=estimate_mask,
            method=method,
            trim_fraction=trim_fraction,
            mode_bins=mode_bins,
            sigma_clip_nsigma=sigma_clip_nsigma,
            sigma_clip_max_iter=sigma_clip_max_iter,
            quantile_q=quantile_q,
        )
        if len(stats_df) == 0:
            score = np.nan
            n_good = 0
            p95_abs_offset = np.nan
        else:
            good = (stats_df["n_pixels"].to_numpy() >= min_pixels_per_line) & np.isfinite(stats_df["line_stat"].to_numpy())
            offsets = stats_df["line_stat"].to_numpy(dtype=float)[good] - global_stat
            counts = stats_df["n_pixels"].to_numpy(dtype=float)[good]
            n_good = int(np.sum(good))
            score = weighted_robust_std(offsets, weights=counts)
            p95_abs_offset = float(np.nanpercentile(np.abs(offsets), 95)) if offsets.size > 0 else np.nan

        rows.append({
            "direction": direction,
            "angle_deg": float(angle_deg),
            "slope": float(slope),
            "score_robust_std_of_line_offsets": score,
            "p95_abs_line_offset": p95_abs_offset,
            "n_good_lines": n_good,
            "global_stat": float(global_stat),
        })

    score_df = pd.DataFrame(rows)
    if np.all(~np.isfinite(score_df["score_robust_std_of_line_offsets"].to_numpy())):
        best_angle = float(base_angle_deg)
        best_slope = angle_to_slope_runtime(best_angle)
    else:
        idx = int(np.nanargmax(score_df["score_robust_std_of_line_offsets"].to_numpy()))
        best_angle = float(score_df.iloc[idx]["angle_deg"])
        best_slope = float(score_df.iloc[idx]["slope"])
    return best_slope, best_angle, score_df


def destripe_by_directional_lines(
    alpha_map: np.ndarray,
    valid_mask: Optional[np.ndarray] = None,
    exclude_mask: Optional[np.ndarray] = None,
    direction: str = "y_minus_x",
    slope: Optional[float] = None,
    line_bin_width: float = 1.0,
    method: str = "median",
    min_pixels_per_line: int = 5,
    preserve_global_stat: bool = True,
    smooth_half_window: int = 0,
    fallback_to_valid: bool = True,
    trim_fraction: float = 0.1,
    mode_bins: int = 64,
    sigma_clip_nsigma: float = 3.0,
    sigma_clip_max_iter: int = 3,
    quantile_q: float = 0.5,
) -> dict:
    """Estimate oriented stripe offsets and subtract them from alpha_map.

    This version supports slopes near y=x or y=-x:
        row - slope * col = const
    """
    alpha = np.asarray(alpha_map, dtype=float)
    if alpha.ndim != 2:
        raise ValueError("alpha_map must be 2D.")

    finite = np.isfinite(alpha)
    if valid_mask is None:
        valid = finite.copy()
    else:
        valid = np.asarray(valid_mask, dtype=bool) & finite

    if not np.any(valid):
        raise ValueError("No valid finite pixels for destriping.")

    if exclude_mask is None:
        estimate_mask = valid.copy()
    else:
        estimate_mask = valid & (~np.asarray(exclude_mask, dtype=bool))

    if not np.any(estimate_mask):
        if fallback_to_valid:
            estimate_mask = valid.copy()
        else:
            raise ValueError("No pixels remain after exclusion for stripe estimation.")

    if slope is None:
        slope = default_slope_for_direction(direction)
    slope = float(slope)

    ids = line_id_map(alpha.shape, direction=direction, slope=slope, line_bin_width=line_bin_width)
    id_min = int(np.nanmin(ids[valid]))
    id_max = int(np.nanmax(ids[valid]))
    id_values = np.arange(id_min, id_max + 1, dtype=np.int64)

    global_stat = statistic_1d(
        alpha[estimate_mask],
        method=method,
        trim_fraction=trim_fraction,
        mode_bins=mode_bins,
        sigma_clip_nsigma=sigma_clip_nsigma,
        sigma_clip_max_iter=sigma_clip_max_iter,
        quantile_q=quantile_q,
    )

    stats_est = grouped_line_stats(
        alpha=alpha,
        ids=ids,
        mask=estimate_mask,
        method=method,
        trim_fraction=trim_fraction,
        mode_bins=mode_bins,
        sigma_clip_nsigma=sigma_clip_nsigma,
        sigma_clip_max_iter=sigma_clip_max_iter,
        quantile_q=quantile_q,
    )
    stats_valid = grouped_line_stats(
        alpha=alpha,
        ids=ids,
        mask=valid,
        method=method,
        trim_fraction=trim_fraction,
        mode_bins=mode_bins,
        sigma_clip_nsigma=sigma_clip_nsigma,
        sigma_clip_max_iter=sigma_clip_max_iter,
        quantile_q=quantile_q,
    ) if fallback_to_valid else pd.DataFrame(columns=["line_id", "n_pixels", "line_stat"])

    raw_stats = np.full(id_values.shape, np.nan, dtype=float)
    counts_used = np.zeros(id_values.shape, dtype=int)
    used_fallback = np.zeros(id_values.shape, dtype=bool)

    # Fill from estimate mask.
    for _, row in stats_est.iterrows():
        line_id = int(row["line_id"])
        if id_min <= line_id <= id_max:
            idx = line_id - id_min
            raw_stats[idx] = float(row["line_stat"])
            counts_used[idx] = int(row["n_pixels"])

    # Fallback to all valid pixels if exclusion left too few pixels.
    if fallback_to_valid and len(stats_valid) > 0:
        valid_lookup = {int(r.line_id): (int(r.n_pixels), float(r.line_stat)) for r in stats_valid.itertuples(index=False)}
        for idx, line_id in enumerate(id_values):
            if counts_used[idx] < min_pixels_per_line:
                if int(line_id) in valid_lookup:
                    count_f, stat_f = valid_lookup[int(line_id)]
                    if count_f >= min_pixels_per_line:
                        counts_used[idx] = count_f
                        raw_stats[idx] = stat_f
                        used_fallback[idx] = True

    # Mark too-short lines as NaN so they are not used.
    raw_stats[counts_used < min_pixels_per_line] = np.nan

    smoothed_stats = moving_nanmedian_1d(raw_stats, half_window=smooth_half_window)

    if preserve_global_stat and np.isfinite(global_stat):
        offsets = smoothed_stats - global_stat
    else:
        offsets = smoothed_stats.copy()

    # Too-short or all-NaN lines are left uncorrected.
    offsets_filled = np.where(np.isfinite(offsets), offsets, 0.0)

    # Fast map from line_id to offset using indexing.
    stripe_map = np.full_like(alpha, np.nan, dtype=float)
    offset_lookup = np.zeros(id_max - id_min + 1, dtype=float)
    offset_lookup[:] = offsets_filled
    valid_ids = np.clip(ids - id_min, 0, offset_lookup.size - 1)
    stripe_map[valid] = offset_lookup[valid_ids[valid]]
    stripe_map[~valid] = np.nan

    corrected = alpha.copy()
    corrected[valid] = alpha[valid] - stripe_map[valid]
    corrected[~finite] = np.nan

    line_table = pd.DataFrame({
        "line_id": id_values,
        "n_pixels_used": counts_used,
        "used_fallback_to_valid": used_fallback,
        "line_stat_raw": raw_stats,
        "line_stat_after_smoothing": smoothed_stats,
        "stripe_offset_subtracted": offsets_filled,
        "global_stat": global_stat,
        "direction": direction,
        "slope": slope,
        "line_bin_width": float(line_bin_width),
        "method": method,
    })

    return {
        "corrected": corrected,
        "stripe_map": stripe_map,
        "line_table": line_table,
        "global_stat": global_stat,
        "estimate_mask": estimate_mask,
        "slope": slope,
        "line_bin_width": float(line_bin_width),
    }


def destripe_by_sequential_directions(
    alpha_map: np.ndarray,
    valid_mask: np.ndarray,
    plume_mask: Optional[np.ndarray] = None,
    destripe_params: Optional[dict] = None,
    nsigma: float = 4.0,
) -> dict:
    """Apply near-diagonal destriping sequentially.

    Example:
        directions=["y_minus_x", "y_plus_x"]

    This means:
        1. remove stripes parallel or nearly parallel to y=x
        2. from the corrected result, remove stripes parallel or nearly parallel to y=-x
    """
    if destripe_params is None:
        destripe_params = {}

    directions = normalize_directions(destripe_params)
    if len(directions) == 0:
        zero = np.zeros_like(alpha_map, dtype=float)
        return {
            "corrected": np.asarray(alpha_map, dtype=float).copy(),
            "stripe_map": zero,
            "directional_stripe_maps": {},
            "line_table": pd.DataFrame(),
            "exclude_meta": [],
        }

    current = np.asarray(alpha_map, dtype=float).copy()
    total_stripe = np.zeros_like(current, dtype=float)
    directional_stripe_maps: dict[str, np.ndarray] = {}
    line_tables = []
    exclude_metas = []

    recompute_exclude = destripe_params.get("recompute_exclude_each_direction", True)
    fixed_exclude_mask = None
    fixed_exclude_meta = None

    if not recompute_exclude:
        fixed_exclude_mask, fixed_exclude_meta = make_exclude_mask_for_destriping(
            alpha_map=current,
            valid_mask=valid_mask,
            plume_mask=plume_mask,
            exclude_mode=destripe_params.get("exclude_mode", "robust_high"),
            exclude_nsigma=destripe_params.get("exclude_nsigma", nsigma),
        )

    for pass_index, direction in enumerate(directions, start=1):
        if recompute_exclude:
            exclude_mask, exclude_meta = make_exclude_mask_for_destriping(
                alpha_map=current,
                valid_mask=valid_mask,
                plume_mask=plume_mask,
                exclude_mode=destripe_params.get("exclude_mode", "robust_high"),
                exclude_nsigma=destripe_params.get("exclude_nsigma", nsigma),
            )
        else:
            exclude_mask = fixed_exclude_mask
            exclude_meta = dict(fixed_exclude_meta)

        slope = get_line_slope(destripe_params, direction)
        line_bin_width = float(destripe_params.get("line_bin_width", 1.0))

        out = destripe_by_directional_lines(
            alpha_map=current,
            valid_mask=valid_mask,
            exclude_mask=exclude_mask,
            direction=direction,
            slope=slope,
            line_bin_width=line_bin_width,
            method=destripe_params.get("method", "median"),
            min_pixels_per_line=destripe_params.get("min_pixels_per_line", 5),
            preserve_global_stat=destripe_params.get("preserve_global_stat", True),
            smooth_half_window=destripe_params.get("smooth_half_window", 0),
            fallback_to_valid=destripe_params.get("fallback_to_valid", True),
            trim_fraction=destripe_params.get("trim_fraction", 0.1),
            mode_bins=destripe_params.get("mode_bins", 64),
            sigma_clip_nsigma=destripe_params.get("sigma_clip_nsigma", 3.0),
            sigma_clip_max_iter=destripe_params.get("sigma_clip_max_iter", 3),
            quantile_q=destripe_params.get("quantile_q", 0.5),
        )

        current = out["corrected"]
        # Treat invalid pixels as zero contribution to the total stripe map.
        total_stripe = np.where(np.isfinite(total_stripe), total_stripe, 0.0) + np.where(np.isfinite(out["stripe_map"]), out["stripe_map"], 0.0)
        total_stripe[~np.asarray(valid_mask, dtype=bool)] = np.nan
        directional_stripe_maps[direction] = out["stripe_map"].copy()

        table = out["line_table"].copy()
        table.insert(0, "pass_index", pass_index)
        line_tables.append(table)

        exclude_meta = dict(exclude_meta)
        exclude_meta.update({
            "pass_index": pass_index,
            "direction": direction,
            "slope": slope,
            "line_bin_width": line_bin_width,
        })
        exclude_metas.append(exclude_meta)

    line_table_all = pd.concat(line_tables, ignore_index=True) if len(line_tables) > 0 else pd.DataFrame()

    return {
        "corrected": current,
        "stripe_map": total_stripe,
        "directional_stripe_maps": directional_stripe_maps,
        "line_table": line_table_all,
        "exclude_meta": exclude_metas,
    }



In [ ]:
# ============================================
# 6. Iterative MF with optional destriping
# ============================================

def should_apply_destriping(iteration_number: int, destripe_when) -> bool:
    if destripe_when is None or destripe_when == "none":
        return False
    if destripe_when == "each_iter":
        return True
    if destripe_when == "final_only":
        return False
    if isinstance(destripe_when, (list, tuple, set, np.ndarray)):
        return int(iteration_number) in {int(v) for v in destripe_when}
    raise ValueError("destripe_when must be 'none', 'each_iter', 'final_only', or a list of iteration numbers.")


def run_iterative_mf_with_optional_destriping(
    cube: np.ndarray,
    uas: np.ndarray,
    valid_mask: Optional[np.ndarray] = None,
    initial_background_mask: Optional[np.ndarray] = None,
    n_iter: int = 5,
    nsigma: float = 4.0,
    reg: float = 1e-6,
    rcond: float = 1e-8,
    min_background_pixels: Optional[int] = None,
    destripe_when = "none",
    destripe_params: Optional[dict] = None,
    verbose: bool = True,
) -> dict:
    """Run Iterative MF, optionally inserting sequential directional destriping into the loop."""
    H, W, B = cube.shape

    if valid_mask is None:
        valid_mask = np.all(np.isfinite(cube), axis=2)
    valid_mask = np.asarray(valid_mask, dtype=bool)

    if initial_background_mask is None:
        background_mask = valid_mask.copy()
    else:
        background_mask = valid_mask & np.asarray(initial_background_mask, dtype=bool)

    if min_background_pixels is None:
        min_background_pixels = max(B + 5, 30)

    if destripe_params is None:
        destripe_params = {}

    alpha_raw_history = []
    alpha_corrected_history = []
    alpha_used_history = []
    stripe_history = []
    directional_stripe_history = []
    plume_mask_history = []
    background_mask_history = []
    threshold_meta_history = []
    line_table_history = []
    exclude_meta_history = []
    mu_history = []
    cov_history = []

    prev_plume_mask = None
    converged_iter = None

    directions = normalize_directions(destripe_params) if destripe_params else []

    for it in range(1, n_iter + 1):
        n_bg = int(np.sum(background_mask))
        if n_bg < min_background_pixels:
            raise ValueError(f"Background pixels too few at iter {it}: {n_bg} < {min_background_pixels}")

        alpha_raw, mu, cov, target = matched_filter_alpha_map(
            cube=cube,
            uas=uas,
            valid_mask=valid_mask,
            background_mask=background_mask,
            reg=reg,
            rcond=rcond,
            min_background_pixels=min_background_pixels,
        )

        alpha_corrected = alpha_raw.copy()
        alpha_used = alpha_raw.copy()
        stripe_map = np.zeros((H, W), dtype=float)
        directional_stripe_maps = {}
        line_table = pd.DataFrame()
        exclude_meta = []

        apply_now = should_apply_destriping(it, destripe_when)
        if apply_now:
            out = destripe_by_sequential_directions(
                alpha_map=alpha_raw,
                valid_mask=valid_mask,
                plume_mask=prev_plume_mask,
                destripe_params=destripe_params,
                nsigma=nsigma,
            )

            alpha_corrected = out["corrected"]
            stripe_map = out["stripe_map"]
            directional_stripe_maps = out["directional_stripe_maps"]
            line_table = out["line_table"]
            exclude_meta = out["exclude_meta"]

            threshold_source = destripe_params.get("threshold_source", "corrected")
            if threshold_source == "corrected":
                alpha_used = alpha_corrected.copy()
            elif threshold_source == "raw":
                alpha_used = alpha_raw.copy()
            else:
                raise ValueError("threshold_source must be 'corrected' or 'raw'.")

        plume_mask, threshold_meta = plume_mask_from_alpha(alpha_used, valid_mask, nsigma=nsigma)
        new_background_mask = valid_mask & (~plume_mask)

        alpha_raw_history.append(alpha_raw.copy())
        alpha_corrected_history.append(alpha_corrected.copy())
        alpha_used_history.append(alpha_used.copy())
        stripe_history.append(stripe_map.copy())
        directional_stripe_history.append({k: v.copy() for k, v in directional_stripe_maps.items()})
        plume_mask_history.append(plume_mask.copy())
        background_mask_history.append(background_mask.copy())
        threshold_meta_history.append(threshold_meta.copy())
        line_table_history.append(line_table.copy())
        exclude_meta_history.append(exclude_meta.copy() if hasattr(exclude_meta, 'copy') else exclude_meta)
        mu_history.append(mu.copy())
        cov_history.append(cov.copy())

        if verbose:
            print(
                f"iter {it:02d} | bg={n_bg:6d} | "
                f"thr={threshold_meta['threshold']:+.6e} | "
                f"med={threshold_meta['median']:+.6e} | "
                f"rstd={threshold_meta['robust_std']:.6e} | "
                f"plume={threshold_meta['n_plume']:6d} | "
                f"destripe={apply_now} | directions={directions if apply_now else []}"
            )

        if prev_plume_mask is not None and np.array_equal(plume_mask, prev_plume_mask):
            converged_iter = it
            background_mask = new_background_mask
            if verbose:
                print(f"Converged at iteration {it}.")
            break

        prev_plume_mask = plume_mask.copy()
        background_mask = new_background_mask

    # Final-only destriping: apply after the iterative background/mask loop.
    final_only_post = None
    if destripe_when == "final_only":
        final_raw = alpha_raw_history[-1]
        final_loop_plume = plume_mask_history[-1]

        out = destripe_by_sequential_directions(
            alpha_map=final_raw,
            valid_mask=valid_mask,
            plume_mask=final_loop_plume,
            destripe_params=destripe_params,
            nsigma=nsigma,
        )

        final_corr = out["corrected"]
        final_plume, final_meta = plume_mask_from_alpha(final_corr, valid_mask, nsigma=nsigma)

        alpha_corrected_history[-1] = final_corr.copy()
        alpha_used_history[-1] = final_corr.copy()
        stripe_history[-1] = out["stripe_map"].copy()
        directional_stripe_history[-1] = {k: v.copy() for k, v in out["directional_stripe_maps"].items()}
        plume_mask_history[-1] = final_plume.copy()
        threshold_meta_history[-1] = final_meta.copy()
        line_table_history[-1] = out["line_table"].copy()
        exclude_meta_history[-1] = out["exclude_meta"]

        final_only_post = {
            "threshold_meta": final_meta,
            "exclude_meta": out["exclude_meta"],
        }

    result = {
        "alpha_raw_history": alpha_raw_history,
        "alpha_corrected_history": alpha_corrected_history,
        "alpha_used_history": alpha_used_history,
        "stripe_history": stripe_history,
        "directional_stripe_history": directional_stripe_history,
        "plume_mask_history": plume_mask_history,
        "background_mask_history": background_mask_history,
        "threshold_meta_history": threshold_meta_history,
        "line_table_history": line_table_history,
        "exclude_meta_history": exclude_meta_history,
        "mu_history": mu_history,
        "cov_history": cov_history,
        "valid_mask": valid_mask,
        "background_mask_final": background_mask,
        "converged_iter": converged_iter,
        "destripe_when": destripe_when,
        "destripe_params": destripe_params,
        "final_only_post": final_only_post,
    }

    # Convenience aliases
    result["alpha_final_raw"] = alpha_raw_history[-1]
    result["alpha_final_corrected"] = alpha_corrected_history[-1]
    result["alpha_final_used"] = alpha_used_history[-1]
    result["stripe_map_final"] = stripe_history[-1]
    result["directional_stripe_maps_final"] = directional_stripe_history[-1]
    result["plume_mask_final"] = plume_mask_history[-1]
    result["threshold_meta_final"] = threshold_meta_history[-1]
    result["line_table_final"] = line_table_history[-1]

    return result



In [ ]:
# ============================================
# 7. Result comparison / saving helpers
# ============================================

def get_result_directions(res: dict):
    params = res.get("destripe_params")
    if params is None:
        return None
    return normalize_directions(params)


def summarize_results_table(results: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for name, res in results.items():
        meta = res["threshold_meta_final"]
        params = res.get("destripe_params")
        rows.append({
            "name": name,
            "destripe_when": res["destripe_when"],
            "method": None if params is None else params.get("method"),
            "directions": None if params is None else " -> ".join(normalize_directions(params)),
            "line_slopes": None if params is None else str(params.get("line_slopes")),
            "line_bin_width": None if params is None else params.get("line_bin_width"),
            "smooth_half_window": None if params is None else params.get("smooth_half_window"),
            "exclude_mode": None if params is None else params.get("exclude_mode"),
            "converged_iter": res["converged_iter"],
            "threshold": meta["threshold"],
            "median": meta["median"],
            "robust_std": meta["robust_std"],
            "plume_pixels": int(np.sum(res["plume_mask_final"])),
            "valid_pixels": int(np.sum(res["valid_mask"])),
        })
    return pd.DataFrame(rows)


def plot_single_result(res: dict, title_prefix: str = "result"):
    valid = res["valid_mask"]
    raw = res["alpha_final_raw"]
    corr = res["alpha_final_corrected"]
    stripe = res["stripe_map_final"]
    plume = res["plume_mask_final"]
    directional_maps = res.get("directional_stripe_maps_final", {})

    vmin, vmax = robust_limits([raw, corr], mask=valid, q_low=2, q_high=98)
    svmin, svmax = robust_limits(stripe, mask=valid, q_low=2, q_high=98)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

    im0 = axes[0].imshow(raw, origin="upper", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"{title_prefix}\nraw alpha")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("y")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, label="alpha")

    im1 = axes[1].imshow(stripe, origin="upper", vmin=svmin, vmax=svmax)
    axes[1].set_title("total estimated stripe")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("y")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, label="offset")

    im2 = axes[2].imshow(corr, origin="upper", vmin=vmin, vmax=vmax)
    axes[2].set_title("corrected alpha")
    axes[2].set_xlabel("x")
    axes[2].set_ylabel("y")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, label="alpha")

    im3 = axes[3].imshow(plume, origin="upper")
    axes[3].set_title("plume mask")
    axes[3].set_xlabel("x")
    axes[3].set_ylabel("y")
    plt.colorbar(im3, ax=axes[3], fraction=0.046, label="candidate")

    plt.tight_layout()
    plt.show()

    # Direction-wise stripe maps
    if len(directional_maps) > 0:
        n = len(directional_maps)
        fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5))
        if n == 1:
            axes = [axes]
        for ax, (direction, smap) in zip(axes, directional_maps.items()):
            lo, hi = robust_limits(smap, mask=valid, q_low=2, q_high=98)
            im = ax.imshow(smap, origin="upper", vmin=lo, vmax=hi)
            ax.set_title(f"stripe: {direction}")
            ax.set_xlabel("x")
            ax.set_ylabel("y")
            plt.colorbar(im, ax=ax, fraction=0.046, label="offset")
        plt.tight_layout()
        plt.show()

    line_table = res["line_table_final"]
    if line_table is not None and len(line_table) > 0:
        plt.figure(figsize=(8, 4))
        for direction, df_dir in line_table.groupby("direction"):
            plt.plot(
                df_dir["line_id"],
                df_dir["stripe_offset_subtracted"],
                marker=".",
                linewidth=1,
                label=direction,
            )
        plt.axhline(0, color="black", linewidth=1)
        plt.xlabel("line_id")
        plt.ylabel("subtracted offset")
        plt.title(f"{title_prefix}: directional stripe offset")
        plt.grid(True)
        plt.legend()
        plt.show()


def plot_experiment_grid(results: dict[str, dict], names: Optional[Sequence[str]] = None):
    if names is None:
        names = list(results.keys())

    valid = next(iter(results.values()))["valid_mask"]
    maps = []
    for name in names:
        maps.append(results[name]["alpha_final_raw"])
        maps.append(results[name]["alpha_final_corrected"])
    vmin, vmax = robust_limits(maps, mask=valid, q_low=2, q_high=98)

    n = len(names)
    fig, axes = plt.subplots(3, n, figsize=(4 * n, 12))
    if n == 1:
        axes = axes.reshape(3, 1)

    for j, name in enumerate(names):
        res = results[name]
        raw = res["alpha_final_raw"]
        corr = res["alpha_final_corrected"]
        plume = res["plume_mask_final"]

        im0 = axes[0, j].imshow(raw, origin="upper", vmin=vmin, vmax=vmax)
        axes[0, j].set_title(f"{name}\nraw")
        axes[0, j].set_xlabel("x")
        axes[0, j].set_ylabel("y")

        im1 = axes[1, j].imshow(corr, origin="upper", vmin=vmin, vmax=vmax)
        axes[1, j].set_title("corrected")
        axes[1, j].set_xlabel("x")
        axes[1, j].set_ylabel("y")

        im2 = axes[2, j].imshow(plume, origin="upper")
        axes[2, j].set_title("plume mask")
        axes[2, j].set_xlabel("x")
        axes[2, j].set_ylabel("y")

    fig.colorbar(im1, ax=axes[0:2, :].ravel().tolist(), fraction=0.02, label="alpha")
    plt.tight_layout()
    plt.show()



def stat_case_name(method: str, when: str) -> str:
    return f"{method}_yx_then_ynegx_{when}"



def available_stat_methods(results: dict[str, dict], methods: Optional[Sequence[str]] = None) -> list[str]:
    if methods is None:
        methods = [m for m, _ in STRIPE_STAT_METHODS]
    out = []
    for method in methods:
        if stat_case_name(method, "final_only") in results or stat_case_name(method, "each_iter") in results:
            out.append(method)
    return out



def plot_when_comparison_by_stat(
    results: dict[str, dict],
    when: str,
    methods: Optional[Sequence[str]] = None,
    include_baseline: bool = True,
):
    """Show one overview image for final_only or each_iter across all statistics."""
    if when not in {"final_only", "each_iter"}:
        raise ValueError("when must be 'final_only' or 'each_iter'.")

    methods = available_stat_methods(results, methods)
    names = [stat_case_name(method, when) for method in methods if stat_case_name(method, when) in results]
    if include_baseline and "baseline_no_destripe" in results:
        names = ["baseline_no_destripe"] + names

    if len(names) == 0:
        print(f"No cases found for {when}.")
        return

    print(f"Overview: {when} / " + ", ".join(names))
    plot_experiment_grid(results, names=names)



def plot_stat_method_results_separately(
    results: dict[str, dict],
    methods: Optional[Sequence[str]] = None,
    include_pair_grid: bool = True,
    show_single_result: bool = True,
):
    """For each statistic, show final_only and each_iter as separate detailed images."""
    methods = available_stat_methods(results, methods)

    for method in methods:
        names = [
            stat_case_name(method, "final_only"),
            stat_case_name(method, "each_iter"),
        ]
        names = [name for name in names if name in results]
        if len(names) == 0:
            continue

        print("\n" + "-" * 80)
        print(f"Statistic: {method}")
        print("-" * 80)

        if include_pair_grid and len(names) >= 2:
            plot_experiment_grid(results, names=names)

        if show_single_result:
            for name in names:
                plot_single_result(results[name], title_prefix=name)



def plot_stat_threshold_histories(results: dict[str, dict], methods: Optional[Sequence[str]] = None):
    """Show threshold histories for final_only and each_iter, grouped by statistic."""
    methods = available_stat_methods(results, methods)
    plt.figure(figsize=(10, 6))
    for method in methods:
        for when, linestyle in [("final_only", "--"), ("each_iter", "-")]:
            name = stat_case_name(method, when)
            if name not in results:
                continue
            th = [m["threshold"] for m in results[name]["threshold_meta_history"]]
            plt.plot(np.arange(1, len(th) + 1), th, marker="o", linestyle=linestyle, label=name)
    plt.xlabel("Iteration")
    plt.ylabel("Threshold")
    plt.title("Threshold history by statistic")
    plt.grid(True)
    plt.legend(fontsize=8)
    plt.show()



def plot_threshold_histories(results: dict[str, dict]):
    plt.figure(figsize=(8, 5))
    for name, res in results.items():
        th = [m["threshold"] for m in res["threshold_meta_history"]]
        plt.plot(np.arange(1, len(th) + 1), th, marker="o", label=name)
    plt.xlabel("Iteration")
    plt.ylabel("Threshold")
    plt.title("Threshold history")
    plt.grid(True)
    plt.legend()
    plt.show()


def save_case_outputs(
    result: dict,
    case_name: str,
    output_dir: str | Path,
    ys: Optional[np.ndarray] = None,
    xs: Optional[np.ndarray] = None,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    raw = result["alpha_final_raw"]
    corr = result["alpha_final_corrected"]
    stripe = result["stripe_map_final"]
    plume = result["plume_mask_final"]
    valid = result["valid_mask"]
    directional_maps = result.get("directional_stripe_maps_final", {})

    np.save(output_dir / f"{case_name}_alpha_raw.npy", raw)
    np.save(output_dir / f"{case_name}_alpha_corrected.npy", corr)
    np.save(output_dir / f"{case_name}_stripe_map_total.npy", stripe)
    np.save(output_dir / f"{case_name}_plume_mask.npy", plume)

    for direction, smap in directional_maps.items():
        np.save(output_dir / f"{case_name}_stripe_map_{direction}.npy", smap)

    H, W = raw.shape
    rows, cols = np.indices((H, W))

    if ys is not None and len(ys) == H:
        y_values = np.asarray(ys)[rows.ravel()]
    else:
        y_values = rows.ravel()

    if xs is not None and len(xs) == W:
        x_values = np.asarray(xs)[cols.ravel()]
    else:
        x_values = cols.ravel()

    y_minus_x_ids = line_id_map(raw.shape, direction="y_minus_x")
    y_plus_x_ids = line_id_map(raw.shape, direction="y_plus_x")

    pixel_df = pd.DataFrame({
        "row": rows.ravel(),
        "col": cols.ravel(),
        "y": y_values,
        "x": x_values,
        "line_id_y_minus_x": y_minus_x_ids.ravel(),
        "line_id_y_plus_x": y_plus_x_ids.ravel(),
        "is_valid": valid.ravel(),
        "alpha_raw": raw.ravel(),
        "stripe_offset_total_subtracted": stripe.ravel(),
        "alpha_corrected": corr.ravel(),
        "is_plume": plume.ravel(),
    })

    for direction, smap in directional_maps.items():
        pixel_df[f"stripe_offset_{direction}"] = smap.ravel()

    pixel_df.to_csv(output_dir / f"{case_name}_pixel_results.csv", index=False)

    line_table = result["line_table_final"]
    if line_table is not None and len(line_table) > 0:
        line_table.to_csv(output_dir / f"{case_name}_line_table.csv", index=False)

    return pixel_df




In [ ]:

# ============================================
# 8. Load data and prepare cube/UAS
# ============================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load ROI spectra
roi_df, wavelengths, spectra = load_roi_spectra_csv(ROI_CSV)
cube, ys, xs = spectra_to_cube(roi_df, spectra)

print(f"ROI table shape: {roi_df.shape}")
print(f"Cube shape: {cube.shape}")
print(f"Wavelength range: {wavelengths[0]:.2f} - {wavelengths[-1]:.2f} nm")

# Quick visual checks
plot_mean_spectrum(cube, wavelengths, xlim=(WL_MIN, WL_MAX))

try:
    rgb = make_rgb_from_cube(cube, wavelengths)
    plt.figure(figsize=(5, 5))
    plt.imshow(rgb, origin="upper")
    plt.title("RGB preview")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()
except Exception as exc:
    print(f"RGB preview skipped: {exc}")

# Load MODTRAN and compute UAS
mod_wave, alpha_grid, mod_spectra = load_ch4_modtran_csv(MODTRAN_CSV)
print(f"MODTRAN wavelength range: {mod_wave[0]:.2f} - {mod_wave[-1]:.2f} nm")
print("alpha_grid:", alpha_grid)

mod_sensor = gaussian_srf_resample(
    mod_wave=mod_wave,
    mod_spectra=mod_spectra,
    sensor_wave=wavelengths,
    fwhm_nm=FWHM_NM,
)

uas_all, intercept = compute_uas_log_slope(
    alpha_grid=alpha_grid,
    spectra_grid=mod_sensor,
    alpha_min=UAS_ALPHA_MIN,
    alpha_max=UAS_ALPHA_MAX,
)

plot_uas(wavelengths, uas_all, title="CH4 UAS from MODTRAN", xlim=(WL_MIN, WL_MAX))

# Select SWIR / methane absorption range
cube_sel, wave_sel, band_sel = select_bands(cube, wavelengths, wl_min=WL_MIN, wl_max=WL_MAX)
uas_sel = uas_all[band_sel]

valid_mask = make_valid_pixel_mask(
    cube_sel,
    nodata_values=NODATA_VALUES,
    require_positive=REQUIRE_POSITIVE,
    min_valid_fraction=MIN_VALID_FRACTION,
)

print(f"Selected cube shape: {cube_sel.shape}")
print(f"Valid pixels: {int(np.sum(valid_mask))}")

plot_map(valid_mask.astype(float), title="Valid pixel mask", cmap="gray", colorbar_label="valid")


In [ ]:
# ============================================
# 9A. Read HISUI metadata and set geometry-guided stripe angles
# ============================================
# このセルは、HISUI L1G の metadata txt がある場合、
# Observation corners を Map image 上の pixel coordinate に変換して、
# 観測 line / sample 方向の角度を推定します。
#
# 推定した角度は DEFAULT_DESTRIPE_PARAMS に反映され、
# 次のセルの angle search の初期中心になります。

METADATA_TXT = Path("HSHL1G_N320W1032_20221030160051_20231127193053.txt")

# True: metadata angle を中心に、さらに alpha map から角度を少しだけ自動補正する
# False: metadata angle をそのまま使う
REFINE_METADATA_ANGLES_FROM_ALPHA = True

# metadata angle 周りの探索幅。
# 見た目でさらにずれているなら 15〜25 deg くらいまで広げる。
METADATA_ANGLE_SEARCH_HALF_RANGE_DEG = {
    "y_minus_x": 12.0,
    "y_plus_x": 12.0,
}

# line bin 幅。角度は合っているのに細かく残る場合は 1.5 or 2.0 を試す。
METADATA_LINE_BIN_WIDTH = 1.0


def parse_key_value_metadata(path: str | Path) -> dict:
    """Parse HISUI-like key=value metadata txt."""
    path = Path(path)
    meta = {}
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"')
            try:
                if value.upper() == "N/A":
                    meta[key] = value
                else:
                    meta[key] = float(value)
            except Exception:
                meta[key] = value
    return meta


def normalize_line_angle_deg(angle_deg: float) -> float:
    """Normalize an undirected line angle to [-90, 90) degrees."""
    angle = (float(angle_deg) + 90.0) % 180.0 - 90.0
    return angle


def circular_mean_undirected_deg(angles_deg):
    """Mean of undirected line angles with 180 deg periodicity."""
    angles = np.asarray([normalize_line_angle_deg(a) for a in angles_deg], dtype=float)
    theta = np.deg2rad(2.0 * angles)
    mean_theta = np.arctan2(np.mean(np.sin(theta)), np.mean(np.cos(theta))) / 2.0
    return normalize_line_angle_deg(np.rad2deg(mean_theta))


def compute_hisui_metadata_line_angles(path: str | Path) -> tuple[dict, pd.DataFrame]:
    """Compute observation line/sample directions in final image coordinates.

    Returns:
        angles: dict with y_minus_x and y_plus_x angles in image coordinates.
                x=col rightward, y=row downward.
        corner_df: corner pixel positions and diagnostic values.
    """
    meta = parse_key_value_metadata(path)

    H = int(meta["ImageLines"])
    W = int(meta["ImageSamples"])
    utm_zone = int(meta.get("UTMZone", 13))

    def latlon(prefix):
        return float(meta[f"{prefix}LatitudeDegree"]), float(meta[f"{prefix}LongitudeDegree"])

    # Projection: prefer UTM because MapProjection=UTM. Fallback to lon/lat affine if pyproj is unavailable.
    try:
        from pyproj import Transformer
        epsg = 32600 + utm_zone  # northern hemisphere; this scene is in N latitude
        transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)

        def project(lat, lon):
            x, y = transformer.transform(lon, lat)
            return np.array([x, y], dtype=float)

        projection_used = f"UTM EPSG:{epsg}"
    except Exception as exc:
        print(f"pyproj unavailable; using lon/lat affine fallback. Reason: {exc}")

        def project(lat, lon):
            return np.array([lon, lat], dtype=float)

        projection_used = "lon/lat affine fallback"

    map_ul = project(*latlon("MapUpperLeft"))
    map_ur = project(*latlon("MapUpperRight"))
    map_ll = project(*latlon("MapLowerLeft"))

    # world = map_ul + col * v_col + row * v_row
    v_col = (map_ur - map_ul) / (W - 1)
    v_row = (map_ll - map_ul) / (H - 1)
    A = np.column_stack([v_col, v_row])
    Ainv = np.linalg.inv(A)

    def world_to_pixel(prefix):
        xy = project(*latlon(prefix))
        col, row = Ainv @ (xy - map_ul)
        return float(row), float(col)

    corners = {
        "ObservationUpperLeft": world_to_pixel("ObservationUpperLeft"),
        "ObservationUpperRight": world_to_pixel("ObservationUpperRight"),
        "ObservationLowerLeft": world_to_pixel("ObservationLowerLeft"),
        "ObservationLowerRight": world_to_pixel("ObservationLowerRight"),
    }

    def segment_angle(p1, p2):
        r1, c1 = corners[p1]
        r2, c2 = corners[p2]
        drow = r2 - r1
        dcol = c2 - c1
        slope = drow / dcol
        angle = np.rad2deg(np.arctan2(drow, dcol))
        return {
            "segment": f"{p1}->{p2}",
            "row1": r1,
            "col1": c1,
            "row2": r2,
            "col2": c2,
            "drow": drow,
            "dcol": dcol,
            "slope_drow_dcol": slope,
            "angle_deg_raw": angle,
            "angle_deg_line": normalize_line_angle_deg(angle),
        }

    # Observation line / along-track-like direction: left edge and right edge.
    along_segments = [
        segment_angle("ObservationUpperLeft", "ObservationLowerLeft"),
        segment_angle("ObservationUpperRight", "ObservationLowerRight"),
    ]

    # Observation sample / cross-track-like direction: top edge and bottom edge.
    cross_segments = [
        segment_angle("ObservationUpperLeft", "ObservationUpperRight"),
        segment_angle("ObservationLowerLeft", "ObservationLowerRight"),
    ]

    along_angle = circular_mean_undirected_deg([s["angle_deg_line"] for s in along_segments])
    cross_angle = circular_mean_undirected_deg([s["angle_deg_line"] for s in cross_segments])

    out_df = pd.DataFrame(along_segments + cross_segments)
    out_df["projection_used"] = projection_used
    out_df["ImageLines"] = H
    out_df["ImageSamples"] = W

    angles = {
        "y_minus_x": float(along_angle),
        "y_plus_x": float(cross_angle),
    }
    return angles, out_df


if METADATA_TXT.exists():
    metadata_angles, metadata_geometry_df = compute_hisui_metadata_line_angles(METADATA_TXT)
    metadata_slopes = {k: angle_to_slope_runtime(v) for k, v in metadata_angles.items()}

    print("Metadata-derived stripe angles [deg]:")
    print(metadata_angles)
    print("Metadata-derived row/col slopes:")
    print(metadata_slopes)
    display(metadata_geometry_df)

    # Use metadata angles as priors/centers.
    DEFAULT_DESTRIPE_PARAMS["line_angles_deg"] = metadata_angles.copy()
    DEFAULT_DESTRIPE_PARAMS["line_slopes"] = metadata_slopes.copy()
    DEFAULT_DESTRIPE_PARAMS["line_bin_width"] = METADATA_LINE_BIN_WIDTH

    # Refine around metadata angles, rather than around ideal 45/-45 deg.
    DEFAULT_DESTRIPE_PARAMS["auto_estimate_line_angles"] = REFINE_METADATA_ANGLES_FROM_ALPHA
    DEFAULT_DESTRIPE_PARAMS["auto_estimate_line_slopes"] = REFINE_METADATA_ANGLES_FROM_ALPHA
    DEFAULT_DESTRIPE_PARAMS["angle_search_half_range_deg"] = METADATA_ANGLE_SEARCH_HALF_RANGE_DEG
    DEFAULT_DESTRIPE_PARAMS["do_fine_angle_search"] = True

    print("Updated DEFAULT_DESTRIPE_PARAMS line_angles_deg:")
    print(DEFAULT_DESTRIPE_PARAMS["line_angles_deg"])
    print("Updated angle_search_half_range_deg:")
    print(DEFAULT_DESTRIPE_PARAMS["angle_search_half_range_deg"])
else:
    print(f"Metadata txt not found: {METADATA_TXT}")
    print("Continue with the default/wide angle search settings.")



In [ ]:
# ============================================
# 9. Estimate stripe angles and run experiments
# ============================================

# ------------------------------------------------------------------
# Optional: estimate angles from the baseline Iterative MF alpha map.
# ------------------------------------------------------------------
angle_score_tables = {}
slope_score_tables = angle_score_tables  # backwards-compatible name

if DEFAULT_DESTRIPE_PARAMS.get("auto_estimate_line_angles", DEFAULT_DESTRIPE_PARAMS.get("auto_estimate_line_slopes", False)):
    print("\n" + "=" * 80)
    print("Estimating stripe angles from baseline Iterative MF alpha map")
    print("=" * 80)

    baseline_for_angle = run_iterative_mf_with_optional_destriping(
        cube=cube_sel,
        uas=uas_sel,
        valid_mask=valid_mask,
        initial_background_mask=None,
        n_iter=N_ITER,
        nsigma=NSIGMA,
        reg=REG,
        rcond=RCOND,
        min_background_pixels=None,
        destripe_when="none",
        destripe_params=None,
        verbose=False,
    )

    alpha_for_angle = baseline_for_angle["alpha_final_raw"]
    estimated_angles = dict(DEFAULT_DESTRIPE_PARAMS.get("line_angles_deg", INITIAL_LINE_ANGLES_DEG.copy()))
    estimated_slopes = dict(DEFAULT_DESTRIPE_PARAMS.get("line_slopes", INITIAL_LINE_SLOPES.copy()))

    # For angle estimation, usually do not exclude high-alpha pixels because the stripe itself
    # may be high-alpha. If real plume dominates the angle estimate, change this to robust_high.
    angle_exclude_mode = DEFAULT_DESTRIPE_PARAMS.get("slope_estimate_exclude_mode", "none")
    angle_exclude_mask, angle_exclude_meta = make_exclude_mask_for_destriping(
        alpha_map=alpha_for_angle,
        valid_mask=valid_mask,
        plume_mask=None,
        exclude_mode=angle_exclude_mode,
        exclude_nsigma=DEFAULT_DESTRIPE_PARAMS.get("exclude_nsigma", NSIGMA),
    )
    print("Angle-estimation exclude meta:", angle_exclude_meta)

    for direction in normalize_directions(DEFAULT_DESTRIPE_PARAMS):
        base_angle = float(estimated_angles.get(direction, slope_to_angle_runtime(estimated_slopes.get(direction, default_slope_for_direction(direction)))))
        half_angle = float(get_direction_param(
            DEFAULT_DESTRIPE_PARAMS,
            "angle_search_half_range_deg",
            direction,
            30.0,
        ))
        steps = int(DEFAULT_DESTRIPE_PARAMS.get("angle_search_steps", 121))

        best_slope, best_angle, score_df = estimate_best_angle_for_direction(
            alpha_map=alpha_for_angle,
            valid_mask=valid_mask,
            direction=direction,
            base_angle_deg=base_angle,
            search_half_range_deg=half_angle,
            search_steps=steps,
            line_bin_width=float(DEFAULT_DESTRIPE_PARAMS.get("line_bin_width", 1.0)),
            method=DEFAULT_DESTRIPE_PARAMS.get("method", "median"),
            min_pixels_per_line=DEFAULT_DESTRIPE_PARAMS.get("min_pixels_per_line", 5),
            exclude_mask=angle_exclude_mask,
            trim_fraction=DEFAULT_DESTRIPE_PARAMS.get("trim_fraction", 0.1),
            mode_bins=DEFAULT_DESTRIPE_PARAMS.get("mode_bins", 64),
            sigma_clip_nsigma=DEFAULT_DESTRIPE_PARAMS.get("sigma_clip_nsigma", 3.0),
            sigma_clip_max_iter=DEFAULT_DESTRIPE_PARAMS.get("sigma_clip_max_iter", 3),
            quantile_q=DEFAULT_DESTRIPE_PARAMS.get("quantile_q", 0.5),
        )

        # Optional fine search around the coarse best angle.
        if DEFAULT_DESTRIPE_PARAMS.get("do_fine_angle_search", True):
            fine_half = float(DEFAULT_DESTRIPE_PARAMS.get("fine_angle_search_half_range_deg", 3.0))
            fine_steps = int(DEFAULT_DESTRIPE_PARAMS.get("fine_angle_search_steps", 61))
            fine_slope, fine_angle, fine_df = estimate_best_angle_for_direction(
                alpha_map=alpha_for_angle,
                valid_mask=valid_mask,
                direction=direction,
                base_angle_deg=best_angle,
                search_half_range_deg=fine_half,
                search_steps=fine_steps,
                line_bin_width=float(DEFAULT_DESTRIPE_PARAMS.get("line_bin_width", 1.0)),
                method=DEFAULT_DESTRIPE_PARAMS.get("method", "median"),
                min_pixels_per_line=DEFAULT_DESTRIPE_PARAMS.get("min_pixels_per_line", 5),
                exclude_mask=angle_exclude_mask,
                trim_fraction=DEFAULT_DESTRIPE_PARAMS.get("trim_fraction", 0.1),
                mode_bins=DEFAULT_DESTRIPE_PARAMS.get("mode_bins", 64),
                sigma_clip_nsigma=DEFAULT_DESTRIPE_PARAMS.get("sigma_clip_nsigma", 3.0),
                sigma_clip_max_iter=DEFAULT_DESTRIPE_PARAMS.get("sigma_clip_max_iter", 3),
                quantile_q=DEFAULT_DESTRIPE_PARAMS.get("quantile_q", 0.5),
            )
            fine_df["search_stage"] = "fine"
            score_df["search_stage"] = "coarse"
            score_df = pd.concat([score_df, fine_df], ignore_index=True)
            best_slope, best_angle = fine_slope, fine_angle

        estimated_angles[direction] = best_angle
        estimated_slopes[direction] = best_slope
        angle_score_tables[direction] = score_df

        print(
            f"{direction}: base_angle={base_angle:+.3f} deg, "
            f"best_angle={best_angle:+.3f} deg, best_slope={best_slope:+.6f}"
        )
        display(score_df.sort_values("score_robust_std_of_line_offsets", ascending=False).head(15))

        # Plot angle score curve.
        plt.figure(figsize=(7, 4))
        for stage, group in score_df.groupby("search_stage") if "search_stage" in score_df.columns else [("search", score_df)]:
            plt.plot(
                group["angle_deg"],
                group["score_robust_std_of_line_offsets"],
                marker="o",
                ms=3,
                label=stage,
            )
        plt.axvline(best_angle, linestyle="--", label=f"best={best_angle:.3f} deg")
        plt.xlabel("stripe angle [deg]")
        plt.ylabel("alignment score")
        plt.title(f"Angle search score: {direction}")
        plt.grid(True)
        plt.legend()
        plt.show()

    # Update default params and all experiment params with the estimated angles/slopes.
    DEFAULT_DESTRIPE_PARAMS["line_angles_deg"] = estimated_angles.copy()
    DEFAULT_DESTRIPE_PARAMS["line_slopes"] = estimated_slopes.copy()
    for cfg in EXPERIMENTS.values():
        if cfg.get("destripe_params") is not None:
            cfg["destripe_params"]["line_angles_deg"] = estimated_angles.copy()
            cfg["destripe_params"]["line_slopes"] = estimated_slopes.copy()

    print("Estimated line angles [deg]:", estimated_angles)
    print("Estimated line slopes:", estimated_slopes)
else:
    print("Automatic angle estimation is disabled.")
    print("Using line angles:", DEFAULT_DESTRIPE_PARAMS.get("line_angles_deg"))
    print("Using line slopes:", DEFAULT_DESTRIPE_PARAMS.get("line_slopes"))

# ------------------------------------------------------------------
# Run all experiments.
# ------------------------------------------------------------------
results = {}

for name, cfg in EXPERIMENTS.items():
    print("\n" + "=" * 80)
    print(f"Running experiment: {name}")
    print("=" * 80)

    res = run_iterative_mf_with_optional_destriping(
        cube=cube_sel,
        uas=uas_sel,
        valid_mask=valid_mask,
        initial_background_mask=None,
        n_iter=N_ITER,
        nsigma=NSIGMA,
        reg=REG,
        rcond=RCOND,
        min_background_pixels=None,
        destripe_when=cfg.get("destripe_when", "none"),
        destripe_params=cfg.get("destripe_params", None),
        verbose=True,
    )
    results[name] = res

summary_df = summarize_results_table(results)
summary_df


In [ ]:
# ============================================
# 10. Compare results
# ============================================

# Summary table.
summary_df = summarize_results_table(results)
display(summary_df)

# Threshold histories for all experiments.
plot_threshold_histories(results)

# Threshold histories grouped by statistic and timing.
plot_stat_threshold_histories(results)

# One overview image for final_only across statistics.
plot_when_comparison_by_stat(results, when="final_only")

# One overview image for each_iter across statistics.
plot_when_comparison_by_stat(results, when="each_iter")

# For each statistic, show final_only and each_iter as separate detailed images.
# This includes median, mean, trimmed_mean, mode, and sigma_clipped_mean.
plot_stat_method_results_separately(
    results,
    methods=[m for m, _ in STRIPE_STAT_METHODS],
    include_pair_grid=True,
    show_single_result=True,
)

# Reference comparison: y=x only vs y=x -> y=-x.
primary_names = [
    "baseline_no_destripe",
    "median_yx_final_only",
    "median_yx_then_ynegx_final_only",
    "median_yx_each_iter",
    "median_yx_then_ynegx_each_iter",
]
primary_names = [name for name in primary_names if name in results]
plot_experiment_grid(results, names=primary_names)


In [ ]:
# ============================================
# 11. Save selected outputs
# ============================================

# Save every displayed statistic for both final_only and each_iter.
cases_to_save = [
    "baseline_no_destripe",
    "median_yx_final_only",
    "median_yx_each_iter",
]

for method, _ in STRIPE_STAT_METHODS:
    cases_to_save.extend([
        f"{method}_yx_then_ynegx_final_only",
        f"{method}_yx_then_ynegx_each_iter",
    ])

for case_name in cases_to_save:
    if case_name in results:
        save_case_outputs(
            result=results[case_name],
            case_name=case_name,
            output_dir=OUTPUT_DIR,
            ys=ys,
            xs=xs,
        )
        print(f"Saved: {case_name}")
    else:
        print(f"Skipped missing case: {case_name}")


## 使い方の目安

### 1. 角度探索範囲を広げる

線が `y=x` から大きくずれている場合は、設定セルの以下を大きくしてください。

```python
ANGLE_SEARCH_HALF_RANGE_DEG = {
    "y_minus_x": 30.0,
    "y_plus_x": 30.0,
}
```

たとえば `45±30°` なら `15〜75°` を探索します。もっと広げるなら `40.0` などにします。ただし広げすぎると plume や地表構造に引っ張られやすくなります。

### 2. 自動推定が怪しい場合

自動推定を止めて手動指定できます。

```python
AUTO_ESTIMATE_LINE_ANGLES = False
INITIAL_LINE_ANGLES_DEG = {
    "y_minus_x": 52.0,
    "y_plus_x": -42.0,
}
```

### 3. 残る線が太い・分裂する場合

`LINE_BIN_WIDTH` を `1.5` または `2.0` にします。

```python
LINE_BIN_WIDTH = 1.5
```

ただし大きくしすぎると、本物の plume まで削りやすくなります。

### 4. まず見るべき結果

`median_yx_then_ynegx_final_only` と `sigma_clipped_mean_yx_then_ynegx_final_only` を最初に見てください。線が background 更新を壊していそうなら、対応する `each_iter` も見ます。
